In [1]:
import tomotopy as tp
import numpy as np
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec
from gensim.models import KeyedVectors
from operator import itemgetter
import matplotlib.pyplot as plt
import pandas as pd
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
import openai
from googletrans import Translator
import matplotlib.pyplot as plt

In [2]:
top_n = 10

In [3]:
def get_word_embedding_based_similarity(topic1, topic2):
    similarity = 0
    panalty = 0
    for i in [word[0] for word in hlda_model.get_topic_words(topic1, top_n=top_n)]:
        for j in [word[0] for word in hlda_model.get_topic_words(topic2, top_n=top_n)]:
            #similarity+= word2vec_model.wv.similarity(i,j)
            s = word2vec_model.wv.similarity(i,j)
            similarity+= s
            if s == 1:
                panalty +=1
    if panalty > 5 : similarity /=2
    return similarity/(top_n*top_n)

In [4]:
word2vec_model = Word2Vec.load('./models/word2vec.model')

In [5]:
def get_similarity(topn_dict):
    similarity = 0
    cnt = 0
    for d1_topic in topn_dict:
        similarity+=get_word_embedding_based_similarity(0,d1_topic)
        cnt+=1
        for d2_topic in topn_dict.get(d1_topic):
            similarity+=get_word_embedding_based_similarity(d1_topic, d2_topic)
            cnt+=1
    s = similarity/cnt
    return s

In [6]:
def get_diveristy(topn_dict):
    cnt = len(topn_dict)
    diversity = 0
    for d1_topic in topn_dict:
        diversity += (1-get_word_embedding_based_similarity(0,d1_topic))
        
        if len(topn_dict.get(d1_topic)) == 1: continue
        else : d1_topic_list = topn_dict.get(d1_topic)
        for i in range(len(d1_topic_list)):
            for j in range(i):
                diversity += (1 - get_word_embedding_based_similarity(d1_topic_list[i],d1_topic_list[j]))
                cnt+=1
    return diversity / cnt

In [7]:
score_df = pd.DataFrame(columns=['phase','k','hcsd'])
for phase in [1,2,3,4]:
    input_data = pd.read_feather(f'./Datasets/covid{phase}.ftr')
    hlda_model = tp.HLDAModel.load(f'./models/hlda_{phase}.bin')
    texts =input_data.okt

    dictionary = Dictionary(texts)
    corpus = [dictionary.doc2bow(text) for text in texts]
    
    topic_numdoc = {}

    # parent-child dictionary
    total =0
    for depth1 in hlda_model.children_topics(0):
        for topic in hlda_model.children_topics(depth1):
            total +=hlda_model.num_docs_of_topic(topic)
            topic_numdoc[topic] = hlda_model.num_docs_of_topic(topic)

    #similarity_list = []
    #diversity_list=[]
    #coherence_list=[]
    #hcsd_list=[]
    
    # get coherence socre       
    for topic_num in range(10,55,5):
        res = dict(sorted(topic_numdoc.items(), key=itemgetter(1), reverse=True)[:topic_num])
        topn_dict = {}
        for child in res:
            #print("parents: ",hlda_model.parent_topic(child), " topic : ",child )
            if hlda_model.parent_topic(child) in topn_dict:
                temp = topn_dict.get(hlda_model.parent_topic(child))
                temp.append(child)
                topn_dict[hlda_model.parent_topic(child)] = temp
            else : topn_dict[hlda_model.parent_topic(child)] = [child]  



        topics = []
        topics.append([word[0] for word in hlda_model.get_topic_words(0, top_n=top_n)])
        for d1_topic in topn_dict:
            topics.append([word[0] for word in hlda_model.get_topic_words(d1_topic, top_n=top_n)])
            for d2_topic in topn_dict.get(d1_topic):
                topics.append([word[0] for word in hlda_model.get_topic_words(d2_topic, top_n=top_n)])


        cm = CoherenceModel(topics = topics,
                       #corpus = corpus,
                       dictionary = dictionary,
                       texts = texts,
                       coherence = 'c_npmi')
        # get proposed score

        similarity=get_similarity(topn_dict)
        #similarity_list.append(similarity)
        
        diversity=get_diveristy(topn_dict)
        #diversity_list.append(diversity)
        coherence =cm.get_coherence()
        # coherence nomalization
        coherence = (coherence +1)/2
        #coherence_list.append(coherence)
        
        score = (similarity+diversity+coherence)/3
        #hcsd_list.append(score)
        #temp = pd.DataFrame()
        score_df = score_df.append({'phase':phase,'k':topic_num,'hcsd':score},ignore_index=True)
        print("phase",phase,"topic number:",topic_num,similarity,diversity,coherence,score)
    print()

phase 1 topic number: 10 0.5413201746554113 0.5469297374207348 0.5597817439476611 0.549343885341269
phase 1 topic number: 15 0.5527867110369896 0.5464717848752549 0.5573036496202732 0.5521873818441726
phase 1 topic number: 20 0.5349256407168385 0.5427484929053884 0.5531694197982574 0.543614517806828
phase 1 topic number: 25 0.5468514859915485 0.5360993209889087 0.5533993679770509 0.5454500583191694
phase 1 topic number: 30 0.5596148876183329 0.5024973471952999 0.5514746958729236 0.5378623102288521
phase 1 topic number: 35 0.5536606872237576 0.49869795707810566 0.5498755055802299 0.5340780499606977
phase 1 topic number: 40 0.5500783847347354 0.507628112540383 0.5540084088042403 0.537238302026453
phase 1 topic number: 45 0.5573117843433103 0.49114208482680666 0.5534710265590275 0.5339749652430482
phase 1 topic number: 50 0.5575723257638404 0.48027082402579874 0.5453650809854756 0.5277360769250383

phase 2 topic number: 10 0.5920802080784259 0.4097372487408575 0.5414424988806515 0.5144199

In [8]:
score_df

,phase,k,hcsd
0,1.0,10.0,0.549344
1,1.0,15.0,0.552187
2,1.0,20.0,0.543615
3,1.0,25.0,0.545450
4,1.0,30.0,0.537862
5,1.0,35.0,0.534078
6,1.0,40.0,0.537238
7,1.0,45.0,0.533975
8,1.0,50.0,0.527736
9,2.0,10.0,0.514420


In [9]:
score_df.to_feather('./result/hlda_result.ftr')

# Interpretation

In [10]:
top_n = 30

In [11]:
hlda_model = tp.HLDAModel.load('./models/hlda_1.bin')
topic_num = 15
topic_numdoc = {}

# parent-child dictionary
total =0
for depth1 in hlda_model.children_topics(0):
    for topic in hlda_model.children_topics(depth1):
        total +=hlda_model.num_docs_of_topic(topic)
        topic_numdoc[topic] = hlda_model.num_docs_of_topic(topic)


# get coherence socre       

res = dict(sorted(topic_numdoc.items(), key=itemgetter(1), reverse=True)[:topic_num])
topn_dict = {}
for child in res:
    #print("parents: ",hlda_model.parent_topic(child), " topic : ",child )
    if hlda_model.parent_topic(child) in topn_dict:
        temp = topn_dict.get(hlda_model.parent_topic(child))
        temp.append(child)
        topn_dict[hlda_model.parent_topic(child)] = temp
    else : topn_dict[hlda_model.parent_topic(child)] = [child]  



topics = []
topics.append([word[0] for word in hlda_model.get_topic_words(0, top_n=top_n)])
for d1_topic in topn_dict:
    topics.append([word[0] for word in hlda_model.get_topic_words(d1_topic, top_n=top_n)])
    for d2_topic in topn_dict.get(d1_topic):
        topics.append([word[0] for word in hlda_model.get_topic_words(d2_topic, top_n=top_n)])



topics = []
print("topic0 : ",[word[0] for word in hlda_model.get_topic_words(0, top_n=top_n)])
print()
for d1_topic in topn_dict:
    print("  topic",d1_topic, ": ",[word[0] for word in hlda_model.get_topic_words(d1_topic, top_n=top_n)])
    for d2_topic in topn_dict.get(d1_topic):
        print("       topic",d2_topic, ": ",[word[0] for word in hlda_model.get_topic_words(d2_topic, top_n=top_n)])
    print()


topic0 :  ['부작용', '많다', '문제', '미국', '효과', '한국', '수도', '얘기', '병원', '상황', '이야기', '혈전', '가능하다', '걱정', '거지', '교차접종', '괜찮다', '기사', '의사', '이유', '회사', '선택', '접종자', '정부', '높다', '중이', '나라', '예약', '자체', '현재']

  topic 9 :  ['근육통', '주사', '부위', '타이레놀', '아프다', '증상', '몸살', '두통', '경과', '뻐근하다', '통증', '아픔', '괜찮다', '아침', '미열', '오한', '기운', '심하다', '저녁', '피곤하다', '오후', '새벽', '상태', '어깨', '발열', '멀쩡하다', '독감', '오전', '체온', '컨디션']
       topic 17 :  ['아프다', '괜찮다', '차갑다', '걱정', '힘들다', '타이레놀', '엄마', '멀쩡하다', '친구', '다행', '고생', '주사', '주변', '부작용', '심하다', '증상', '무섭다', '뻐근하다', '별로', '교차접종', '엄청나다', '언니', '많다', '근육통', '후유증', '반응', '건강하다', '아빠', '남편', '긴장']
       topic 301 :  ['아프다', '머리', '온몸', '덥다', '춥다', '띵하다', '이상하다', '밤새', '난리', '이불', '뜨겁다', '멍하다', '침대', '선풍기', '여전하다', '자석', '겨울', '장난', '에어컨', '어지럽다', '무선', '퇴근', '얼굴', '이차', '간만', '귀찮다', '따뜻하다', '바람', '관절', '긴장']
       topic 157 :  ['타이레놀', '준비', '미리', '복용', '아프다', '진통제', '아세트아미노펜', '예정', '해열제', '증상', '약국', '반응', '고기', '계열', '부루펜', '성분', '구비', '엄마', '취침', '두시', '삼겹살

In [12]:
hlda_model = tp.HLDAModel.load('./models/hlda_2.bin')
topic_num = 20
topic_numdoc = {}

# parent-child dictionary
total =0
for depth1 in hlda_model.children_topics(0):
    for topic in hlda_model.children_topics(depth1):
        total +=hlda_model.num_docs_of_topic(topic)
        topic_numdoc[topic] = hlda_model.num_docs_of_topic(topic)


# get coherence socre       

res = dict(sorted(topic_numdoc.items(), key=itemgetter(1), reverse=True)[:topic_num])
topn_dict = {}
for child in res:
    #print("parents: ",hlda_model.parent_topic(child), " topic : ",child )
    if hlda_model.parent_topic(child) in topn_dict:
        temp = topn_dict.get(hlda_model.parent_topic(child))
        temp.append(child)
        topn_dict[hlda_model.parent_topic(child)] = temp
    else : topn_dict[hlda_model.parent_topic(child)] = [child]  



topics = []
topics.append([word[0] for word in hlda_model.get_topic_words(0, top_n=top_n)])
for d1_topic in topn_dict:
    topics.append([word[0] for word in hlda_model.get_topic_words(d1_topic, top_n=top_n)])
    for d2_topic in topn_dict.get(d1_topic):
        topics.append([word[0] for word in hlda_model.get_topic_words(d2_topic, top_n=top_n)])



topics = []
print("topic0 : ",[word[0] for word in hlda_model.get_topic_words(0, top_n=top_n)])
print()
for d1_topic in topn_dict:
    print("  topic",d1_topic, ": ",[word[0] for word in hlda_model.get_topic_words(d1_topic, top_n=top_n)])
    for d2_topic in topn_dict.get(d1_topic):
        print("    topic",d2_topic, ": ",[word[0] for word in hlda_model.get_topic_words(d2_topic, top_n=top_n)])
    print()


topic0 :  ['아프다', '부작용', '괜찮다', '걱정', '증상', '많다', '병원', '친구', '주사', '무섭다', '상태', '완료', '힘들다', '추가', '엄마', '다행', '엄청나다', '타이레놀', '컨디션', '멀쩡하다', '후유증', '주변', '심하다', '잔여백신', '고생', '문제', '교차접종', '수도', '일주일', '차갑다']

  topic 8 :  ['아프다', '타이레놀', '몸살', '괜찮다', '근육통', '두통', '머리', '기운', '저녁', '아침', '새벽', '미열', '심하다', '멀쩡하다', '힘들다', '오한', '당일', '아픔', '종일', '온몸', '첫날', '뻐근하다', '감기', '상태', '피곤하다', '컨디션', '무겁다', '증상', '어지럽다', '출근']
    topic 69 :  ['아프다', '주사', '뻐근하다', '부위', '경과', '아픔', '어깨', '증상', '근육통', '통증', '멀쩡하다', '괜찮다', '근육', '부분', '별다르다', '기분', '팔뚝', '불편하다', '딱하다', '멀쩡함', '왼쪽', '독감', '현재', '완료', '무겁다', '묵직하다', '점점', '별로', '위로', '자리']
    topic 202 :  ['아프다', '차갑다', '괜찮다', '걱정', '무섭다', '힘들다', '멀쩡하다', '별로', '주변', '친구', '많다', '다행', '엄마', '심하다', '두렵다', '보통', '건강하다', '확실하다', '주사', '언니', '사례', '개인', '긴장', '교차접종', '얘기', '동생', '지옥', '수도', '뻐근하다', '지인']
    topic 16 :  ['두통', '근육통', '통증', '부위', '증상', '경과', '미열', '심하다', '주사', '오한', '아픔', '어지럼증', '부작용', '뻐근하다', '발열', '타이레놀', '어깨', '아프다', '피로', '메스꺼움', 

In [13]:
hlda_model = tp.HLDAModel.load('./models/hlda_3.bin')
topic_num = 15
topic_numdoc = {}

# parent-child dictionary
total =0
for depth1 in hlda_model.children_topics(0):
    for topic in hlda_model.children_topics(depth1):
        total +=hlda_model.num_docs_of_topic(topic)
        topic_numdoc[topic] = hlda_model.num_docs_of_topic(topic)


# get coherence socre       

res = dict(sorted(topic_numdoc.items(), key=itemgetter(1), reverse=True)[:topic_num])
topn_dict = {}
for child in res:
    #print("parents: ",hlda_model.parent_topic(child), " topic : ",child )
    if hlda_model.parent_topic(child) in topn_dict:
        temp = topn_dict.get(hlda_model.parent_topic(child))
        temp.append(child)
        topn_dict[hlda_model.parent_topic(child)] = temp
    else : topn_dict[hlda_model.parent_topic(child)] = [child]  



topics = []
topics.append([word[0] for word in hlda_model.get_topic_words(0, top_n=top_n)])
for d1_topic in topn_dict:
    topics.append([word[0] for word in hlda_model.get_topic_words(d1_topic, top_n=top_n)])
    for d2_topic in topn_dict.get(d1_topic):
        topics.append([word[0] for word in hlda_model.get_topic_words(d2_topic, top_n=top_n)])


topics = []
print("topic0 : ",[word[0] for word in hlda_model.get_topic_words(0, top_n=top_n)])
print()
for d1_topic in topn_dict:
    print("  topic",d1_topic, ": ",[word[0] for word in hlda_model.get_topic_words(d1_topic, top_n=top_n)])
    for d2_topic in topn_dict.get(d1_topic):
        print("    topic",d2_topic, ": ",[word[0] for word in hlda_model.get_topic_words(d2_topic, top_n=top_n)])
    print()


topic0 :  ['추가', '부작용', '아프다', '괜찮다', '혈통', '많다', '완료', '병원', '걱정', '친구', '엄마', '예약', '고생', '힘들다', '증상', '주사', '수도', '교차접종', '문제', '상태', '후유증', '다행', '무섭다', '주변', '심하다', '고민', '의사', '얘기', '아마', '멀쩡하다']

  topic 8 :  ['아프다', '혈통', '괜찮다', '머리', '멀쩡하다', '차갑다', '아픔', '힘들다', '주사', '타이레놀', '추가', '별로', '엄청나다', '뻐근하다', '어지럽다', '무섭다', '근육통', '걱정', '다행', '무겁다', '온몸', '어깨', '다행하다', '확실하다', '기운', '겨드랑이', '체질', '멀쩡함', '허리', '말짱하다']
    topic 16 :  ['몸살', '근육통', '두통', '통증', '타이레놀', '추가', '증상', '오한', '부위', '심하다', '뻐근하다', '기운', '아픔', '경과', '겨드랑이', '괜찮다', '미열', '새벽', '저녁', '아침', '주사', '당일', '컨디션', '가슴', '발열', '피곤하다', '오전', '팔뚝', '반응', '살기']
    topic 157 :  ['혈통', '예정', '임도', '고요', '행복하다', '귀엽다', '어감', '슈퍼', '아기', '아쉽다', '아깝다', '맥주', '사랑', '주말', '조종', '보신', '영화', '케이크', '이군', '웃음', '아이스크림', '스티커', '보아', '자식', '기회', '나기', '에이', '큰일', '맛있다', '눈치']
    topic 263 :  ['주사', '부위', '뻐근하다', '증상', '경과', '아프다', '추가', '통증', '아픔', '근육통', '바늘', '완료', '자리', '반응', '피곤하다', '딱하다', '면역체계', '악화', '따끔하다', '마찬가지', '묵직하다', 

In [14]:
hlda_model = tp.HLDAModel.load('./models/hlda_4.bin')
topic_num = 10
topic_numdoc = {}

# parent-child dictionary
total =0
for depth1 in hlda_model.children_topics(0):
    for topic in hlda_model.children_topics(depth1):
        total +=hlda_model.num_docs_of_topic(topic)
        topic_numdoc[topic] = hlda_model.num_docs_of_topic(topic)


# get coherence socre       

res = dict(sorted(topic_numdoc.items(), key=itemgetter(1), reverse=True)[:topic_num])
topn_dict = {}
for child in res:
    #print("parents: ",hlda_model.parent_topic(child), " topic : ",child )
    if hlda_model.parent_topic(child) in topn_dict:
        temp = topn_dict.get(hlda_model.parent_topic(child))
        temp.append(child)
        topn_dict[hlda_model.parent_topic(child)] = temp
    else : topn_dict[hlda_model.parent_topic(child)] = [child]  



topics = []
topics.append([word[0] for word in hlda_model.get_topic_words(0, top_n=top_n)])
for d1_topic in topn_dict:
    topics.append([word[0] for word in hlda_model.get_topic_words(d1_topic, top_n=top_n)])
    for d2_topic in topn_dict.get(d1_topic):
        topics.append([word[0] for word in hlda_model.get_topic_words(d2_topic, top_n=top_n)])



topics = []
print("topic0 : ",[word[0] for word in hlda_model.get_topic_words(0, top_n=top_n)])
print()
for d1_topic in topn_dict:
    print("  topic",d1_topic, ": ",[word[0] for word in hlda_model.get_topic_words(d1_topic, top_n=top_n)])
    for d2_topic in topn_dict.get(d1_topic):
        print("    topic",d2_topic, ": ",[word[0] for word in hlda_model.get_topic_words(d2_topic, top_n=top_n)])
    print()


topic0 :  ['부작용', '추가', '괜찮다', '예약', '혈통', '병원', '많다', '효과', '고민', '아프다', '친구', '문제', '주사', '완료', '작년', '의사', '잔여백신', '독감', '회사', '감염', '확진', '상태', '얘기', '오미크론', '힘들다', '새롭다', '예정', '마음', '변이', '현재']

  topic 8 :  ['아프다', '부작용', '증상', '심하다', '후유증', '고생', '걱정', '멀쩡하다', '병원', '별로', '확진', '엄청나다', '반응', '생리', '심장', '기침', '엄마', '이상하다', '다행', '일주일', '운동', '주변', '면역', '무섭다', '금요일', '가족', '고통', '건강', '약하다', '음성']
    topic 16 :  ['아프다', '근육통', '몸살', '괜찮다', '주사', '통증', '뻐근하다', '두통', '타이레놀', '부위', '힘들다', '증상', '동절기', '심하다', '혈통', '아픔', '머리', '컨디션', '독감', '오한', '기운', '미열', '겨드랑이', '경과', '어깨', '추가접종', '상태', '가볍다', '온몸', '아침']
    topic 138 :  ['혈통', '건강하다', '고맙다', '임도', '응원', '많다', '가족', '다행', '행복하다', '상태', '깜짝', '사랑', '무사하다', '슈퍼항체', '이상반응', '완성', '동네', '별다르다', '의심', '안전하다', '연구', '콧물', '취소', '바이러스', '활동', '무증상', '근무', '영업', '계정', '광고']
    topic 196 :  ['생리', '출혈', '부작용', '생리주기', '많다', '기간', '생리불순', '예정', '영향', '문제', '고요', '정상', '인정', '그다음', '주가', '후회', '입원', '정신', '당황', '여전하다', '자궁', '버전', '보호'